# 규칙이 아닌 의미 기반 문서 분리하기  

# Loader
> 문서를 읽어서 Document 객체 생성

In [2]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

file = "./data/Attention Is All You Need.pdf"
loader = PyMuPDF4LLMLoader(
    file_path=file, mode="page"
)

In [3]:
docs = loader.load()
len(docs)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


15

In [6]:
# 각 페이지의 텍스트 문장들을 하나의 문자열로 결합 
full_text = "\n\n".join(
    doc.page_content for doc in docs
)

len(full_text)

40991

In [8]:
print(full_text[:500])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or

scholarly works.

## **Attention Is All You Need**



**Niki Parmar** _[∗]_
Google Research
```
nikip@google.com

```


**Ashish Vaswani** _[∗]_

Google Brain
```
avaswani@google.com

```

**Llion Jones** _[∗]_
Google Research
```
 llion@google.com

```


**Noam Shazeer** _[∗]_

Google Brain
```
noam@google.com

```


**Jakob Uszkoreit** _[


# Split

## LLM Section 분석 

> 목차 내용에 맞게 문서를 자르자!!

In [11]:
from pydantic import BaseModel, Field 

class Section(BaseModel):
    title: str = Field(description="원본에 존재하는 섹션 제목")
    start_text: str = Field(description="해당 섹션의 시작 문자열")

class ListSection(BaseModel):
    sections: list[Section]

In [12]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=ListSection)

In [14]:
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Section": {"properties": {"title": {"description": "원본에 존재하는 섹션 제목", "title": "Title", "type": "string"}, "start_text": {"description": "해당 섹션의 시작 문자열", "title": "Start Text", "type": "string"}}, "required": ["title", "start_text"], "title": "Section", "type": "object"}}, "properties": {"sections": {"items": {"$ref": "#/$defs/Section"}, "title": "Sections", "type": "array"}}, "required": ["sections"]}
```


In [15]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""다음 논문을 분석하여 Section별 제목과
시작 위치를 추출하세요.

반환 형식:
{format_instructions}

주의사항:
- 원문에 존재하는 제목만 반환하세요.
- start_text는 원본 문자열에 실제 존재해야 합니다.
- 목차가 아닌 본문 내 제목을 기준으로 하세요.

논문 내용:
{full_text}
""",
    input_variables=["full_text"],
    partial_variables={"format_instructions":format_instructions}
)

In [16]:
prompt

PromptTemplate(input_variables=['full_text'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"$defs": {"Section": {"properties": {"title": {"description": "원본에 존재하는 섹션 제목", "title": "Title", "type": "string"}, "start_text": {"description": "해당 섹션의 시작 문자열", "title": "Start Text", "type": "string"}}, "required": ["title", "start_text"], "title": "Section", "type": "object"}}, "properties": {"sections": {"items": {"$ref": "#/$defs/Section"}, "title": "Sections", "type": "array"}}, "required": ["sections"]}\n```'}, template='다음

In [20]:
from langchain_openai import ChatOpenAI 

model = ChatOpenAI(
    model="gpt-6-luna"
)

In [21]:
chain = prompt | model | output_parser

In [22]:
results = chain.invoke({
    "full_text":full_text
})

In [44]:
len(results.sections) # LLM이 만든 전체 목차의 수 

26

In [47]:
results.sections[2].title, results.sections[2].start_text 

('2 Background', '**2** **Background**')

## Section Document 생성

In [ ]:
# 실제 본문에서 시작 위치 탐색
positions = []

for section in results.sections:
    # start: 해당 텍스트의 위치 정보
    start = full_text.find(section.start_text)

    if start == -1:
        raise ValueError(f"제목을 찾지 못했습니다: {section.title}")

    positions.append({
        "title": section.title,
        "start": start
    })

# 위치순 정렬
positions.sort(key=lambda x: x["start"])

In [65]:
from langchain_core.documents import Document

# Section별 Document 생성
section_docs = []

for i, section in enumerate(positions):
    start = section["start"]

    if i + 1 < len(positions):
        end = positions[i + 1]["start"]
    else: # 마지막 목차(section)
        end = len(full_text)

    section_docs.append(
        Document(
            page_content=full_text[start:end].strip(),
            metadata={
                "section": section["title"],
                "source":file
            }
        )
    )

In [66]:
len(section_docs) == len(results.sections)

True

In [67]:
print(f"제목: {section_docs[1].metadata['section']}")
print("="*50)
print(section_docs[1].page_content)

제목: 1 Introduction
**1** **Introduction**


Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].


Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states _ht_, as a function of the previous hidden state _ht−_ 1 and the input for position _t_ . This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in 

## RecursiveCharacterTextSplitter

> rule base(RecursiveCharacterTextSplitter)를 사용해서 문서를 자르자!!

In [68]:
# 하나의 문서 내용이 너무 크기 때문에 Split 필요
max(
    len(doc.page_content) for doc in section_docs
)

7840

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# A/B 테스트를 통해 가장 성능이 좋은 chunk_size를 정의해야함!!!
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # 최대 문서의 길이
    chunk_overlap=100, # 각 문서별 겹치는 글자의 길이 
    length_function=len
)

In [71]:
section_docs_by_splitter = splitter.split_documents(section_docs)

In [72]:
len(section_docs_by_splitter), len(section_docs)

(60, 26)

In [73]:
print(f"제목: {section_docs_by_splitter[1].metadata['section']}")
print("="*50)
print(section_docs_by_splitter[1].page_content)

제목: Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 Englishto-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
